# Random Forests: Decorrelating the Trees

**DSAI 621 — Week 7.** Bagging averages many fully grown trees, each fitted to a bootstrap sample
(ISL 8.2.1). A random forest does the same, with one change: at every split, each tree may only
choose among $m$ predictors drawn at random from the $p$ (ISL 8.2.2). In this notebook you will:

1. watch bagging and random forests on one training set as trees are added, and see that bagging
   stops improving at a higher test error;
2. measure the **squared bias**, **variance** and **tree-to-tree correlation** $\rho$ of a single
   tree, of bagging, and of random forests with several values of $m$, using many simulated training
   sets, to see *why*.

All of the code is written for you: run each cell, read what it does, and change it to try
things out. Short exercises marked **E1**, **E2**, … ask you to stop and look at what you just
made, and answer in words. Exercises marked **T1**, … ask you to change a number in a cell,
rerun, and explain what changed. Each one says which cells to rerun and whether to change the
number back afterwards.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# random number generator for reproducibility
rng = np.random.default_rng(621)

### The data

We simulate data where random forests should help: many predictors that carry **overlapping**
information. There are $p = 10$ predictors, each standard normal, and **every pair has correlation
0.7**. The response depends on the first five only:
$$y = f(x) + \varepsilon, \qquad f(x) = \#\{j \le 5 : x_j > 0\}, \qquad \varepsilon \sim N(0, 2^2).$$
So $f(x)$ counts how many of $x_1, \dots, x_5$ are positive, a whole number from 0 to 5. Predictors
6 to 10 don't appear in $f$, but because they are correlated with the first five, they carry some
information about them. Each training set has $n = 200$ observations, and the noise is large
($\sigma = 2$), so a single tree will be very variable.

`make_data` draws one training set. We also draw one large **test set** of 5000 points, to measure
test error accurately.

In [ ]:
P = 10          # number of predictors
CORR = 0.7      # correlation between every pair of predictors
SIGMA = 2.0     # noise standard deviation
N = 200         # training set size

# Covariance matrix of the predictors: 1 on the diagonal, CORR everywhere else
Sigma_X = np.full((P, P), CORR)
np.fill_diagonal(Sigma_X, 1.0)

# True function: how many of the first five predictors are positive
def f_true(X):
    return (X[:, :5] > 0).sum(axis=1).astype(float)

# Function to simulate a data set of size n
def make_data(n, rng):
    "X is n x P with correlated normal columns; y = f(X) + noise."
    X = rng.multivariate_normal(np.zeros(P), Sigma_X, n)
    y = f_true(X) + rng.normal(0, SIGMA, n)
    return X, y

X, y = make_data(N, rng)              # one training set
X_test, y_test = make_data(5000, rng)  # a large test set
print(X.shape, X_test.shape)
print(np.corrcoef(X[:, :4], rowvar=False).round(2))   # correlations of the first four predictors

---
## 1. Bagging vs random forests on one training set

In sklearn, a random forest is `RandomForestRegressor`, and `max_features` is $m$, the number of
predictors each split may choose from. With `max_features=10` (all $p$ predictors) it is
**bagging**. ISL suggests $m \approx \sqrt p$, which is about 3 here.

The cell below fits three forests of 300 trees each to the training set: bagging ($m = 10$), and
random forests with $m = 3$ and $m = 1$. A forest's prediction is the average of its trees'
predictions, so the average of the **first $B$ trees** is what a forest of $B$ trees would predict.
The plot shows the test MSE of that average as $B$ goes from 1 to 300.

The test MSE includes the irreducible error $\sigma^2 = 4$, which no model can remove. The plot
shows **test MSE $-\ \sigma^2$**, an estimate of the part a model *can* reduce, so that the
differences are easier to see. The dashed line is a single fully grown tree, fitted to all of the
training data.

In [ ]:
B = 300
ms = [10, 3, 1]          # max_features: 10 = bagging

fig, ax = plt.subplots(figsize=(8, 4.5))
styles = {10: "-", 3: "--", 1: ":"}
final = {}
for m in ms:
    forest = RandomForestRegressor(n_estimators=B, max_features=m, random_state=0, n_jobs=-1).fit(X, y)
    # predictions of each tree on the test set: B x 5000
    tree_preds = np.array([tree.predict(X_test) for tree in forest.estimators_])
    # average of the first b trees, for b = 1, ..., B
    running_avg = np.cumsum(tree_preds, axis=0) / np.arange(1, B + 1)[:, None]
    test_mse = ((running_avg - y_test) ** 2).mean(axis=1)
    label = "bagging (m = 10)" if m == P else f"random forest (m = {m})"
    ax.plot(np.arange(1, B + 1), test_mse - SIGMA**2, styles[m], lw=2, label=label)
    final[label] = {"B = 1": test_mse[0], "B = 10": test_mse[9], "B = 50": test_mse[49], "B = 300": test_mse[-1]}

tree = DecisionTreeRegressor(random_state=0).fit(X, y)
tree_mse = np.mean((tree.predict(X_test) - y_test) ** 2)
ax.axhline(tree_mse - SIGMA**2, color="grey", ls="-.", label="single fully grown tree")
ax.set(xscale="log", xlabel="number of trees B", ylabel="test MSE minus $\\sigma^2$",
       title="Adding trees: bagging vs random forests", ylim=(0, 5))
ax.legend()
plt.show()

print(f"single tree test MSE: {tree_mse:.3f}")
display(pd.DataFrame(final).T.round(3))

**E1.** Read the plot and the table. With one tree ($B = 1$), how do the three methods compare with
each other and with the single fully grown tree? As trees are added, which method ends lowest? Look
at bagging between $B = 50$ and $B = 300$: is it still improving? Would adding another 1000 trees to
bagging let it catch up with the random forests?

<details>
<summary>Show answer: E1</summary>

**E1. Answer.** With one tree the three methods are about as bad as the single fully grown tree, whose test MSE is
7.99 (about 4.0 above $\sigma^2$): 8.47 for bagging, 7.93 for $m = 3$ and 7.88 for $m = 1$. One bagged
tree is even a little worse than the single tree, because it is grown on a bootstrap sample, which holds only
about 63% of the distinct observations. As trees are added, all three fall fast, and the random forests
end lowest: at $B = 300$ the test MSE is 4.596 for bagging, 4.368 for $m = 3$ and 4.286 for $m = 1$. Above
$\sigma^2 = 4$ that is 0.60, 0.37 and 0.29, so the forest with $m = 1$ has about half of bagging's reducible error.
Bagging has stopped improving: it goes from 4.654 at $B = 50$ to 4.596 at $B = 300$, and its curve is flat.
Another 1000 trees would not let it catch up, because averaging correlated trees can only bring the variance
down to the floor $\rho\,\sigma_T^2$ (Section 2), and bagging's floor is higher.

</details>

---
## 2. Why: bias, variance and correlation over many training sets

One training set and one test set can't tell us *why* one method beats another. As in the decision
trees lab, we simulate many training sets ($R = 50$) from the same population, fit each method to
each one, and look at its predictions at 500 fixed test points $x$. From those we compute, averaged
over the test points:

- **squared bias**: how far the *average* fit (over training sets) is from $f(x)$;
- **variance**: how much the fit moves from one training set to the next;
- for the forests, the **variance of one tree** and the **correlation $\rho$ between two trees** of
  the same forest, across training sets.

The variance of an average of $B$ trees, each with variance $\sigma_T^2$ and pairwise correlation
$\rho$, is (ESL 15.1)
$$\rho\,\sigma_T^2 + \frac{1 - \rho}{B}\,\sigma_T^2.$$
As $B$ grows the second term disappears, but the first does not: **$\rho\,\sigma_T^2$ is a floor**
that no number of trees can get below. The only way to lower it is to lower $\rho$, which is what a
small $m$ does. (We write $\sigma_T^2$ for the tree variance, to keep it apart from the noise
variance $\sigma^2 = 4$.)

`simulate` does the work. For a forest it also records the variance *among the trees of one
forest*, which is the part of a tree's variance that averaging can remove; the rest is what the
trees share, and $\rho$ is that shared part as a fraction of $\sigma_T^2$. You don't need the
details to do the exercises. The cell takes about 15 seconds.

In [ ]:
R = 50                                          # number of simulated training sets
B_SIM = 100                                     # trees per forest in the simulation
X_grid = make_data(500, np.random.default_rng(1))[0]   # 500 fixed test points
f_grid = f_true(X_grid)

# Function to simulate R training sets, fit a model to each, and compute bias^2, variance and rho
def simulate(m, seed=2):
    "m = None fits a single fully grown tree; otherwise a forest with max_features = m."
    rng_sim = np.random.default_rng(seed)       # same seed: the same R training sets every time
    preds = np.empty((R, len(X_grid)))          # each fit's predictions on the grid
    within = np.empty((R, len(X_grid)))         # variance among the trees of one forest
    for r in range(R):
        Xr, yr = make_data(N, rng_sim)
        if m is None:
            model = DecisionTreeRegressor(random_state=r).fit(Xr, yr)
        else:
            model = RandomForestRegressor(n_estimators=B_SIM, max_features=m, random_state=r, n_jobs=-1).fit(Xr, yr)
            within[r] = np.array([t.predict(X_grid) for t in model.estimators_]).var(axis=0)
        preds[r] = model.predict(X_grid)
    var = preds.var(axis=0, ddof=1)
    # (mean of R fits - f)^2 overstates bias^2 by var / R, so subtract that
    bias2 = np.mean((preds.mean(axis=0) - f_grid) ** 2 - var / R)
    row = {"bias^2": bias2, "variance": var.mean(), "bias^2 + variance": bias2 + var.mean()}
    if m is not None:
        w = within.mean()                       # removable by averaging
        shared = var.mean() - w / B_SIM         # variance of an infinitely large forest
        row.update({"tree variance": shared + w, "rho": shared / (shared + w)})
    return row

rows = {"single tree": simulate(None)}
for m in [10, 5, 3, 2, 1]:
    rows["bagging (m = 10)" if m == P else f"random forest, m = {m}"] = simulate(m)
results = pd.DataFrame(rows).T
display(results.round(3))

**E2.** Compare **bagging** with the random forests. Which column changes most as $m$ goes from 10
down to 1: the squared bias, the variance, or the tree variance? Use the $\rho$ column and the floor
$\rho\,\sigma_T^2$ to explain why the forests' variance is lower than bagging's, even though each of
their trees is at least as variable as a bagged tree.

<details>
<summary>Show answer: E2</summary>

**E2. Answer.** The **variance** changes most: it falls from 0.521 for bagging to 0.238 at $m = 1$, more than halving. The
squared bias rises, from 0.128 to 0.257, and the tree variance hardly moves (4.741 to 4.868, slightly *up*:
each forest tree is a bit more variable than a bagged tree). What falls is $\rho$, from 0.101 to 0.039. The
floor $\rho\,\sigma_T^2$ is $0.101 \times 4.741 = 0.48$ for bagging and $0.039 \times 4.868 = 0.19$ at $m = 1$.
With $B = 100$ trees the second term $(1 - \rho)\sigma_T^2/B$ adds about 0.04 to 0.05, which gives the
variances in the table (0.52 and 0.24). So the forests win not because their trees are better, but because
their trees make *different* errors, which cancel when they are averaged.

</details>

**E3.** Now compare the **single tree** with the random forest with $m = 3$. By what factor does the
variance fall? What happens to the squared bias? Give two reasons a forest of trees can have a
*higher* bias than one fully grown tree, when each tree in it is also fully grown. (Hint: what data
does each tree see, and which predictors may it split on?)

<details>
<summary>Show answer: E3</summary>

**E3. Answer.** The variance falls from 4.713 for the single tree to 0.354 for the forest with $m = 3$, about 13 times
smaller. The squared bias rises from 0.094 to 0.174. That increase, 0.08, is tiny next to the drop in variance,
4.36, so bias$^2$ + variance falls from 4.81 to 0.53. The forest's bias is the bias of one of its trees, and
two things make a forest tree more biased than a tree grown on all the data:

1. **Each tree sees a bootstrap sample**, which holds only about 63% of the distinct observations, so it has
   less information to fit $f$. This is why bagging's squared bias (0.128) is already above the single
   tree's (0.094).
2. **Each split may only choose among $m$ predictors**, so it is sometimes forced to split on a weaker one,
   such as one of $x_6, \dots, x_{10}$, which only stand in for the predictors in $f$. The smaller $m$, the
   more often this happens, which is why the squared bias keeps rising from 0.128 ($m = 10$) to 0.257
   ($m = 1$).

</details>

The cell below plots the squared bias and the variance against $m$, with the single tree's values as
horizontal lines (note the log scale on the vertical axis).

In [ ]:
forests = results.drop(index="single tree")
m_values = [10, 5, 3, 2, 1]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(m_values, forests["bias^2"], "o--", lw=2, label="forest: bias$^2$")
ax.plot(m_values, forests["variance"], "s-", lw=2, label="forest: variance")
ax.plot(m_values, forests["bias^2 + variance"], "D:", lw=2, label="forest: bias$^2$ + variance")
ax.axhline(results.loc["single tree", "bias^2"], color="grey", ls="--", lw=1, label="single tree: bias$^2$")
ax.axhline(results.loc["single tree", "variance"], color="grey", ls="-", lw=1, label="single tree: variance")
ax.set(yscale="log", xlabel="m (max_features)", ylabel="value",
       title="Random forests: bias and variance against m", xticks=m_values)
ax.invert_xaxis()        # bagging on the left, smallest m on the right
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

**E4.** Which $m$ gives the lowest bias$^2$ + variance? Describe the trade-off that $m$ controls, in the
same words you would use for $\lambda$ in ridge regression or the size of a tree. Is ISL's suggestion
$m \approx \sqrt p \approx 3$ the best value here? How would you choose $m$ for real data, where you
can't simulate new training sets?

<details>
<summary>Show answer: E4</summary>

**E4. Answer.** $m = 1$ has the lowest bias$^2$ + variance (0.495), with $m = 2$ very close (0.503) and $m = 3$ at 0.527. Bagging
($m = 10$) is worst, at 0.649. A smaller $m$ decorrelates the trees and lowers the variance, but adds bias,
just as a larger $\lambda$ in ridge regression, or a smaller tree, lowers the variance and adds bias. Here the
variance falls faster than the bias rises all the way down to $m = 1$, but the curve is nearly flat from
$m = 3$ down, so ISL's $m \approx \sqrt p$ is close to the best without being exactly it. On real data,
treat $m$ as a tuning parameter: fit forests for a grid of $m$ values and choose by the **out-of-bag error**
(`oob_score=True`, ISL 8.2.1), which needs no extra fitting, or by cross-validation.

</details>

**T1.** In the cell under **The data**, change `CORR = 0.7` to `CORR = 0.0`, so the predictors are
independent and $x_6, \dots, x_{10}$ are pure noise. Rerun every cell from there down (about 30
seconds). How much do the random forests now improve on bagging? What happens to the squared bias at
$m = 1$, and why? (Hint: with $m = 1$, how often is a split forced to use a predictor that has
nothing to do with $y$?) Change `CORR` back to 0.7 and rerun afterwards.

<details>
<summary>Show answer: T1</summary>

**T1. Answer.** With `CORR = 0.0`, bias$^2$ + variance is 0.722 for bagging and 0.630 at the best $m$ ($m = 3$), so the random
forest improves on bagging by about 13%, against 24% (0.649 to 0.495) with correlated predictors. In Section 1
the gap is small too: 4.646 for bagging, 4.562 for $m = 3$ and 4.598 for $m = 1$ at $B = 300$. At $m = 1$
the squared bias jumps to 0.473 (from 0.257), and $m = 1$ is now worse than $m = 3$ (0.685). With independent
predictors, $x_6, \dots, x_{10}$ are pure noise. With $m = 1$, the one predictor a split may use is one of them
half of the time, and splitting on it does nothing to fit $f$. With `CORR = 0.7` those same predictors were
correlated with $x_1, \dots, x_5$, so a forced split on one of them still helped. Random forests do best
when there are many predictors that carry overlapping information, so that hiding some of them costs little.

</details>

---
## 3. Wrap-up questions

**Q1.** Every tree in a bagged forest is grown on its own bootstrap sample, drawn independently of the
others. Why are the trees' predictions still correlated? Why does a smaller $m$ make them less
correlated?

<details>
<summary>Show answer: Q1</summary>

**Q1. Answer.** All the bootstrap samples are drawn from the **same** training set, so every tree sees nearly the same data,
including the same noise. And with $m = p$ every tree is grown greedily from the same predictors, so the top
splits tend to use the same strong predictors and the trees look alike. Their errors at a point $x$ therefore
move together from one training set to the next. With a small $m$, each split can only use the few
predictors it was dealt, so different trees split on different predictors and end up with different
shapes. Their errors overlap less, and $\rho$ falls: from 0.101 for bagging to 0.039 at $m = 1$.

</details>

**Q2.** A colleague says: "Random forests beat a single tree because they have lower bias: hundreds
of trees can fit much more detail than one." Use your table to correct them.

<details>
<summary>Show answer: Q2</summary>

**Q2. Answer.** The table says the opposite. The single fully grown tree has the **lowest** squared bias, 0.094; every forest
has a higher one, from 0.128 (bagging) to 0.257 ($m = 1$). Averaging trees cannot lower the bias: the forest's
expected prediction is that of one of its trees, and each of those trees is grown on a bootstrap sample and
(for $m < p$) with restricted splits. All of the gain is in the **variance**, which falls from 4.713 for one
tree to between 0.24 and 0.52 for the forests.

</details>

**Q3.** Suppose you had $p = 200$ predictors, only 2 of which affect $y$, all independent of each
other. Would you expect a random forest with $m = \sqrt{200} \approx 14$ to beat bagging? Use T1 and
this calculation: the chance that none of the $m$ predictors drawn for a split is one of the two that
matter is $\binom{198}{14} / \binom{200}{14} \approx 0.86$.

<details>
<summary>Show answer: Q3</summary>

**Q3. Answer.** No. About 86% of splits would have no relevant predictor among their $m = 14$
candidates, so most splits would have to be made on noise. The trees would have a high bias, as at $m = 1$ in T1,
but much worse. Bagging ($m = p$) can always choose one of the two predictors that matter, so it would likely beat a
random forest with $m = 14$. ESL 15.3.4 makes this point. A random forest needs a larger $m$ here, and the
way to find it is again to compare the out-of-bag error over a grid of $m$ values.

</details>